In [ ]:
import pandas as pd
from scipy import stats
import numpy as np
from scipy.stats import false_discovery_control
import os
from plot_tools import *
from scipy.stats import chi2
from snp_analysis_tools_sherlock import *
from scipy.stats import binom
import itertools as it
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot
from glob import glob

hv.extension('bokeh')

In [ ]:
def get_selection_stuff_both(species,inoculumn,thresh=1e-3):
    fname1 = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv3_sel_bootstrapv3/{species}/{inoculumn}_parentboth_info.csv'
    sel_stuff = pd.read_csv(fname1).rename(columns={'mesocosms':'mesocosm'})
    # workflow/report/track_snpsv2_sel_bootstrapv3/100910/AA-AE-mBHI_parentboth_info.csv
    fname1=f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3_both/{species}/{inoculumn}_parentboth_info.csv'
    e003_metadata = pd.read_csv('e003_coalescence_metadata_round4_good.csv').set_index('sample')
    df_both = pd.read_csv(fname1).set_index('sample')
  #  df_both = get_both_dfs(fname1)
    df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
    df_both=df_both.loc[df_both['shifts_med']<.1,:]
    

    df_both_meta_good = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                   e003_metadata.index.values),:]], axis=1).reset_index()


   # print('bef',np.max(df_both_meta_good['actual_med1']- df_both_meta_good['boot_med1']))
    df_both_meta_good['actual_med1']=df_both_meta_good['actual_med']#/(df_both_meta_good['actual_med1']+df_both_meta_good['actual_med2'])
    df_both_meta_good['boot_med1']=df_both_meta_good['boot_med']#/(df_both_meta_good['boot_med1']+df_both_meta_good['boot_med2'])
    df_both_meta_good['boot_high1']=df_both_meta_good['boot_high']#/(df_both_meta_good['boot_high1']+df_both_meta_good['boot_high2'])
    df_both_meta_good['boot_low1']=df_both_meta_good['boot_low']#/(df_both_meta_good['boot_low1']+df_both_meta_good['boot_low2'])
    for col in ['boot_med1','boot_low1','boot_high1',]:
        df_both_meta_good.loc[df_both_meta_good[col]<thresh,col]=thresh
        df_both_meta_good.loc[df_both_meta_good[col]>1-thresh,col]=1-thresh

   # print('aft',np.max(df_both_meta_good['actual_med1']- df_both_meta_good['boot_med1']))
    sel_stuff=sel_stuff.loc[sel_stuff['sample1'].isin(df_both_meta_good['sample'].unique())*sel_stuff['sample2'].isin(df_both_meta_good['sample'].unique()),:]
    
    sel_stuff['type_mesocosm'] = sel_stuff['mesocosm'].transform(lambda x: '-'.join(x.split('-')[1:]))
    sel_stuff['parent_subjects'] = sel_stuff['mesocosm'].transform(lambda x: '-'.join(x.split('-')[1:3]))
    sel_stuff['parent_media'] = sel_stuff['mesocosm'].transform(lambda x: x.split('-')[3])
    sel_stuff['media'] = sel_stuff['mesocosm'].transform(lambda x: x.split('-')[-1])

    return sel_stuff, df_both_meta_good


In [ ]:
df = pd.read_csv('~/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/101346/AA-AF-mBHI_parentboth_info.csv')
df_both = get_selection_stuff_both(101346,'AA-AF-mBHI')
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv3_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
all_dfs = []
p1_b = 0
p2_b = 7
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
     #   try:
        if os.path.exists(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv3_sel_bootstrapv3/{sp}/{ino}_parentboth_info.csv'):
            if not os.path.exists(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3_both/{sp}/{ino}_parentboth_info.csv'):
                continue

            df,df_both=get_selection_stuff_both(sp,ino)#.reset_index()
            
       # except:
        #    continue
            df['species_id']=sp
            df['inoculumn']=ino
            df['species-ino']= sp+'-' + ino
            df['p7']= np.nan
            dfp7 = df.loc[(df['passage1']==p1_b)*((df['passage2']==p2_b)),:]
            df_bothp7 = df_both.loc[df_both['passage']==p2_b,:]
            for meso in dfp7['mesocosm'].unique():
             #   print(df.loc[df['mesocosm']==meso,'p7'])
                dfp7.loc[dfp7['mesocosm']==meso,'p7']=df_bothp7.loc[df_bothp7['mesocosm']==meso,'boot_med'].values
            all_dfs.append(dfp7.reset_index())
            
all_dfs=pd.concat(all_dfs)
print(len(all_dfs['species_id'].unique()))

print(len(all_dfs))

all_dfs = all_dfs.loc[(all_dfs['passage1']==p1_b)*((all_dfs['passage2']==p2_b)),:]
all_dfs['sel_med']=all_dfs['sel_med']#*1/np.log2(200)
all_dfs['sel_coeff']=all_dfs['sel_med']

all_dfs = all_dfs.groupby(['sample1','sample2','mesocosm','inoculumn','type_mesocosm','species-ino','species_id','media']).median(numeric_only=True).reset_index()
all_dfs = all_dfs.loc[~all_dfs['species-ino'].isin(bad_inos),:]
print(len(all_dfs))
all_dfs=all_dfs.loc[all_dfs['species-ino']!='100910-AA-AE-mBHI',:] #bad
len(all_dfs['species_id'].unique())
#all_dfs['sel_med']=all_dfs['sel_med']*1/np.log2(200)

In [ ]:
all_dfs

In [ ]:
mGAMs_max=[]
mGAMs_min=[]
mBHIs_max=[]
mBHIs_min=[]
mBHI_est=[]
mGAM_est=[]
p7s_med_mBHI = []
p7s_med_mGAM = []
sp_inos=[]
pvals=[]

for sp_ino in all_dfs['species-ino'].unique():

    df_sp_ino = all_dfs.loc[all_dfs['species-ino']==sp_ino,:]
    df_sp_ino=df_sp_ino.loc[~df_sp_ino['sel_med'].isna(),:]
    media = df_sp_ino['media'].unique()
    if len(media)<2:
        print(sp_ino, media)
        continue
    mGAMs = df_sp_ino.loc[df_sp_ino['media']=='mGAM','p7'].values
    mBHIs = df_sp_ino.loc[df_sp_ino['media']=='mBHI','p7'].values
    if len(mBHIs) <1 or len(mGAMs) < 1:
        print(sp_ino)
        continue
    if len(mBHIs)==1 and len(mGAMs)==1:
        continue
    
    mGAMs_min.append(np.min(mGAMs))
    mGAMs_max.append(np.max(mGAMs))
    mGAM_est.append(np.median(mGAMs))
 
    print(sp_ino,np.min(mGAMs)>0, np.max(mGAMs)>0, np.median(mGAMs)>0)
    

    mBHIs_min.append(np.min(mBHIs))
    mBHIs_max.append(np.max(mBHIs))
    mBHI_est.append(np.median(mBHIs))

    sp_inos.append(sp_ino)
 #   print(mGAMs)
    thresh = 1e-3
    use_thresh = False
    if use_thresh:
        mGAMs[mGAMs<=thresh]=thresh
       # print(mGAMs)
        
        mGAMs[mGAMs>=1-thresh]=1-thresh
        mBHIs[mBHIs<=thresh]=thresh
        mBHIs[mBHIs>=1-thresh]=1-thresh
    t_statistic, p_value = stats.ttest_ind(mGAMs, mBHIs)
    if np.isnan(p_value):
        print(mGAMs,mBHIs)
  #  print(p_value)
    pvals.append(p_value)


df = pd.DataFrame(data={'mGAM_min':mGAMs_min,'mBHI_min':mBHIs_min,
                        'mGAM_max':mGAMs_max,'mBHI_max':mBHIs_max,
                        'mBHI':mBHI_est,
                        'mGAM':mGAM_est,
                        'pvals':pvals,
                        'species-inoculumn': sp_inos})


df=df.sort_values(by='pvals')
dfno_nan=df.loc[~df['pvals'].isna(),:]
qvals = np.sort(dfno_nan['pvals'].values)
qvals = stats.false_discovery_control(np.sort(qvals[~np.isnan(qvals)]))
dfno_nan['qvals']=qvals
dfno_nan.loc[dfno_nan['qvals']<.05,:]

len(dfno_nan)
print(len(dfno_nan))
print(len(dfno_nan.loc[dfno_nan['qvals']<.05,:]))
print(len(df))
major_sig = dfno_nan.loc[dfno_nan['qvals']<.001,'species-inoculumn'].values
minor_sig = dfno_nan.loc[(dfno_nan['qvals']<.05)*(dfno_nan['qvals']>.001),'species-inoculumn'].values
## this was how we got signififance now we gotta get sel

df_good_sigs_p7 = dfno_nan.copy()

In [ ]:
mGAMs_max=[]
mGAMs_min=[]
mBHIs_max=[]
mBHIs_min=[]
mBHI_est=[]
mGAM_est=[]
p7s_med_mBHI = []
p7s_med_mGAM = []
sp_inos=[]
pvals=[]

for sp_ino in all_dfs['species-ino'].unique():
    df_sp_ino = all_dfs.loc[all_dfs['species-ino']==sp_ino,:]
    df_sp_ino=df_sp_ino.loc[~df_sp_ino['sel_med'].isna(),:]
    media = df_sp_ino['media'].unique()
    if len(media)<2:
       # print(sp_ino, media)
        continue
    mGAMs = df_sp_ino.loc[df_sp_ino['media']=='mGAM','sel_coeff'].values
    mBHIs = df_sp_ino.loc[df_sp_ino['media']=='mBHI','sel_coeff'].values
    if len(mBHIs) <1 or len(mGAMs) < 1:
      #  print(sp_ino)
        continue
    if len(mBHIs)==1 and len(mGAMs)==1:
        continue
    
    mGAMs_min.append(np.min(mGAMs))
    mGAMs_max.append(np.max(mGAMs))
    mGAM_est.append(np.median(mGAMs))

    mBHIs_min.append(np.min(mBHIs))
    mBHIs_max.append(np.max(mBHIs))
    mBHI_est.append(np.median(mBHIs))

    sp_inos.append(sp_ino)
 #   print(mGAMs)
   
    t_statistic, p_value = stats.ttest_ind(mGAMs, mBHIs)
  #  if np.isnan(p_value):
   #     print(mGAMs,mBHIs)
  #  print(p_value)
    #pvals.append(p_value)
   # pvals.append(np.max([p_value1, pvalue2]))
    print(sp_ino,np.min(mGAMs)>0, np.max(mGAMs)>0, p_value)

In [ ]:
df['ratio']=np.nan
df['logratio']=np.nan
df_abun = pd.read_csv('e003_coalescence_metadata_round4_abundances.csv')
df_abun['AC']=df_abun['parent_subjects'].transform(lambda x: 'AC' in x)
df_abun=df_abun.loc[~df_abun['AC'],:]
df_abun.loc[df_abun['relative_abundance']<1e-3,'relative_abundance']=1e-3#.drop(columns='Unnamed:0')

df_abun.head()
#df_abun.loc[df_abun['relative_abundance']==0,'relative_abundance']=1e-4
for sp_ino in df['species-inoculumn'].unique():
    sp,sub1,sub2,_ = sp_ino.split('-')
    meso_s1_mGAM = f'{sub1}-{sub1}-mGAM-mGAM'
    meso_s1_mBHI = f'{sub1}-{sub1}-mBHI-mBHI'
    meso_s2_mGAM = f'{sub2}-{sub2}-mGAM-mGAM'
    meso_s2_mBHI = f'{sub2}-{sub2}-mBHI-mBHI'
    sub1media_mBHI= df_abun.loc[(df_abun['type_mesocosm']==meso_s1_mBHI)*(df_abun['species_id'].astype(str)==sp),'relative_abundance'].values[0]
    sub1media_mGAM= df_abun.loc[(df_abun['type_mesocosm']==meso_s1_mGAM)*(df_abun['species_id'].astype(str)==sp),'relative_abundance'].values[0]
    sub2media_mBHI= df_abun.loc[(df_abun['type_mesocosm']==meso_s2_mBHI)*(df_abun['species_id'].astype(str)==sp),'relative_abundance'].values[0]
    sub2media_mGAM= df_abun.loc[(df_abun['type_mesocosm']==meso_s2_mGAM)*(df_abun['species_id'].astype(str)==sp),'relative_abundance'].values[0]
    ratio = (sub1media_mBHI/sub2media_mBHI)/(sub1media_mGAM/sub2media_mGAM)
    log_ratio = np.log10(sub1media_mBHI/sub2media_mBHI) - np.log10(sub1media_mGAM/sub2media_mGAM)
    df.loc[df['species-inoculumn']==sp_ino,'ratio']=ratio
    df.loc[df['species-inoculumn']==sp_ino,'logratio']=log_ratio
df.head()

In [ ]:
import scipy.stats
df['ratio_mBHImGAM']=df['mBHI']/df['mGAM']
df['diff_mBHImGAM'] = df['mBHI']-df['mGAM']
to_plot = 'diff_mBHImGAM'
p = hv.Scatter(df,vdims = ['diff_mBHImGAM'],kdims = ['ratio']).opts(color=bokeh.palettes.Set2[3][-1], size=10,alpha=.5,
                                                            line_color='black',
                                                                    logx=True,
                                                           #ylim = (-1.1,1.1),
                                                          )
p = hv.render(p)
bokeh.io.show(p)
print(scipy.stats.pearsonr(df['ratio'],df[to_plot],))
#p.yaxis.axis_label = 'Abundance ratios'
#p.xaxis.axis_label = 'Sel diffs'
p.output_backend='svg'
export_plot_pdf(p,'sel_diffs')

In [ ]:
df['species_id']=df['species-inoculumn'].transform(lambda x: x.split('-')[0])
print(len(df['species_id'].unique()))
len(df)

In [ ]:
mGAMs_max=[]
mGAMs_min=[]
mBHIs_max=[]
mBHIs_min=[]
mBHI_est=[]
mGAM_est=[]
p7s_med_mBHI = []
p7s_med_mGAM = []
sp_inos=[]
pvals=[]

for sp_ino in all_dfs['species-ino'].unique():
    df_sp_ino = all_dfs.loc[all_dfs['species-ino']==sp_ino,:]
    df_sp_ino=df_sp_ino.loc[~df_sp_ino['sel_med'].isna(),:]
    media = df_sp_ino['media'].unique()
    if len(media)<2:
       # print(sp_ino, media)
        continue
    mGAMs = df_sp_ino.loc[df_sp_ino['media']=='mGAM','sel_coeff'].values
    mBHIs = df_sp_ino.loc[df_sp_ino['media']=='mBHI','sel_coeff'].values
    if len(mBHIs) <1 or len(mGAMs) < 1:
      #  print(sp_ino)
        continue
    if len(mBHIs)==1 and len(mGAMs)==1:
        continue
    
    mGAMs_min.append(np.min(mGAMs))
    mGAMs_max.append(np.max(mGAMs))
    mGAM_est.append(np.median(mGAMs))

    mBHIs_min.append(np.min(mBHIs))
    mBHIs_max.append(np.max(mBHIs))
    mBHI_est.append(np.median(mBHIs))

    sp_inos.append(sp_ino)
 #   print(mGAMs)
   
    t_statistic, p_value = stats.ttest_ind(mGAMs, mBHIs)
  #  if np.isnan(p_value):
   #     print(mGAMs,mBHIs)
  #  print(p_value)
    #pvals.append(p_value)
   # pvals.append(np.max([p_value1, pvalue2]))
    print(sp_ino,np.min(mGAMs)>0, np.max(mGAMs)>0, p_value)



df = pd.DataFrame(data={'mGAM_min':mGAMs_min,'mBHI_min':mBHIs_min,
                        'mGAM_max':mGAMs_max,'mBHI_max':mBHIs_max,
                        'mBHI':mBHI_est,
                        'mGAM':mGAM_est,
               
                        'species-inoculumn': sp_inos})



In [ ]:

df['repol']=False
df.loc[df['mBHI']<0,'repol']=True
to_repol = df.loc[df['repol'],'species-inoculumn'].unique()
df.loc[df['mBHI']<0,'mGAM']=-df.loc[df['mBHI']<0,'mGAM']
df.loc[df['mBHI']<0,'mBHI']=-df.loc[df['mBHI']<0,'mBHI']
#df = df.loc[df['species-inoculumn']!='100910-AA-AE-mBHI',:]

df['species']=df['species-inoculumn'].transform(lambda x: x.split('-')[0])

df = df.sort_values(by='mBHI',ascending=False)
p = hv.Points(df, vdims=['mGAM'],kdims=['species-inoculumn','mGAM'], label='mGAM',).opts(width=500,
                                                                                                     size=5,
                                                                                                color=bokeh.palettes.Set2[8][0],
                                                                                               line_color='black',
                                                                                                 xrotation=60)





p2=hv.Points(df, vdims=['mBHI'],kdims=['species-inoculumn','mBHI'],label='mBHI',).opts(width=500,
                                                                                                    size=5,
                                                                                                   line_color='black',
                                                                                    color=bokeh.palettes.Set2[8][1],
                                                                                                              legend_position='right',
                                                                                                xrotation=60)


all_dfs['species-inoculumn']=all_dfs['species-ino']
all_dfs_p= all_dfs.loc[all_dfs['species-inoculumn'].isin(df['species-inoculumn'].values),:]
all_dfs_p.loc[all_dfs_p['species-inoculumn'].isin(to_repol),'sel_coeff']= -all_dfs_p.loc[all_dfs_p['species-inoculumn'].isin(to_repol),'sel_coeff']

all_dfs_p['species']=all_dfs_p['species-inoculumn'].transform(lambda x: x.split('-')[0])

p3 = hv.Points(all_dfs_p.loc[all_dfs_p['media']=='mGAM',:], vdims=['sel_coeff'],kdims=['species-inoculumn','sel_coeff'],).opts(width=500,
                                                                                                    size=5,
                                                                                    color=bokeh.palettes.Set2[8][0],
                                                                                          alpha=.5,
                                                                                                xrotation=60)

p4 = hv.Points(all_dfs_p.loc[all_dfs_p['media']=='mBHI',:], vdims=['sel_coeff'],kdims=['species-inoculumn','sel_coeff']).opts(width=500,
                                                                                                    size=5,
                                                                                    color=bokeh.palettes.Set2[8][1],
                                                                                          alpha=.5,
                                                                                                xrotation=60)

dfno_nan = df_good_sigs_p7.copy()
print(len(dfno_nan.loc[dfno_nan['qvals']<.05,:]))
print(len(df))
major_sig = dfno_nan.loc[dfno_nan['qvals']<.001,'species-inoculumn'].values
minor_sig = dfno_nan.loc[(dfno_nan['qvals']<.05)*(dfno_nan['qvals']>.001),'species-inoculumn'].values
## this was how we got signififance now we gotta get sel


dfsig = df.loc[df['species-inoculumn'].isin(minor_sig),:].copy()


#dfsig  = dfsig.loc[dfsig['species_ino'].isin(major_sig),:]
dfsig['sel_coeff']= -1.3
dfsig['sel_coeff']= -1.3
dfsig['media'] = 'p<.05'
p5 = hv.Points(dfsig, vdims=['sel_coeff'],kdims=['species-inoculumn','sel_coeff'], label = 'p<.05').opts(width=500,marker='*',color = 'black',
                                                                                                    size=5,
                                                                                )

dfsigsig = df.loc[df['species-inoculumn'].isin(major_sig),:]
dfsigsig['sel_coeff']= -1.3
dfsigsig['sel_coeff']= -1.3
dfsigsig['media'] = 'p<.05'
p6 = hv.Points(dfsigsig, vdims=['sel_coeff'],kdims=['species-inoculumn','sel_coeff'],  label = 'p<.001').opts(width=350,marker='+',color = 'black',
                                                                                                    size=5,
                                                                                )
p=hv.render(p*p2*p3*p4*p2*p5*p6)
p.yaxis.axis_label='Sel Coeff'
p.xaxis.axis_label='Competition'

p.ray(x=0,y=0,angle=0,color='black')
p.output_backend = "svg"
p.y_range = bokeh.models.Range1d(-1.5,1.5)
#p.y_range = bokeh.models.Range1d(-.8,.8)
test_name='bloh'
bokeh.io.export_svgs(p, filename = test_name + '.svg')
export_plot_pdf(p,f'media_stuff_{str(p1_b)}_{str(p2_b)}')
bokeh.io.show(p)

In [ ]:
list(minor_sig)+list(major_sig)

In [ ]:
#['100099-AE-AF-mBHI', '101294-AA-AF-mGAM', '101294-AE-AF-mGAM',
       #'100099-AA-AF-mGAM', '100099-AE-AF-mGAM', '100146-AA-AE-mGAM',
       #'102478-AA-AF-mGAM', '101346-AA-AF-mGAM', '100099-AA-AE-mGAM',
       #'100146-AE-AF-mGAM']

In [ ]:
df_abun_good = df_abun.loc[df_abun['passage']==0,:]
df_abun_good.head()

In [ ]:
import itertools as it

In [ ]:
df_abun_p0 = df_abun.loc[df_abun['passage'] == 0,:]
df_abun_p0 =df_abun_p0.loc[df_abun_p0['relative_abundance']>.05,:]
df_abun_p0 =df_abun_p0.loc[df_abun_p0['parent_subjects'].isin(['AA-AA','AE-AE','AF-AF']),:]
samp1 = []
samp2 =[]
host1=[]
host2=[]
med1=[]
med2=[]
shared_sp = []
for s1,s2 in it.combinations(df_abun_p0['sample'].unique(),2):
    s1sp = df_abun_p0.loc[df_abun_p0['sample']==s1,'species_id'].values
    s2sp = df_abun_p0.loc[df_abun_p0['sample']==s1,'species_id'].values
    shared_sp.append(len(np.intersect1d(s1sp,s2sp)))
    samp1.append(s1)
    samp2.append(s2)
    med1.append( df_abun_p0.loc[df_abun_p0['sample']==s1,'media'].values[0])
    med2.append( df_abun_p0.loc[df_abun_p0['sample']==s2,'media'].values[0])
    host1.append( df_abun_p0.loc[df_abun_p0['sample']==s1,'parent_subjects'].values[0])
    host2.append( df_abun_p0.loc[df_abun_p0['sample']==s2,'parent_subjects'].values[0])
df = pd.DataFrame(data={'samp1':samp1,'samp2':samp2,'host1':host1,'host2':host2,'med1':med1,'med2':med2,'shared_sp':shared_sp})
dfgood = df.loc[df['med1']==df['med2'],:]
dfgood